# Summary Workflow — n8n Batch Runner

Sends each row of `summary_test_case.xlsx` to the **Summary Workflow** webhook and appends
`result_subject`, `result_objective`, `result_debt_cause`, `result_offer_suitability`,
`result_request_information`, and `result_summary` columns.
Output is saved to `summary_result.xlsx`.

> The test-case file already stores payload fields in the **flattened** shape the webhook expects,
> so each row maps directly to the request body without further transformation.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
N8N_BASE_URL  = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH  = "515736a7-e9eb-4ea0-81cb-bfd4a4248a8b"
USE_TEST_URL  = False   # True → /webhook-test/…  |  False → /webhook/…

INPUT_FILE    = "summary_test_case.xlsx"
OUTPUT_FILE   = "summary_result.xlsx"

TIMEOUT       = 60      # seconds per request (summary agent is slower)
RETRIES       = 2       # extra attempts on failure
DELAY_BETWEEN = 0.5     # seconds between rows

In [2]:
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

In [3]:
def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


def _str(val, default: str = "") -> str:
    return str(val) if pd.notna(val) else default


def _num(val, default=0):
    try:
        return float(val) if pd.notna(val) else default
    except (TypeError, ValueError):
        return default


def build_payload(row: dict) -> dict:
    """Map a spreadsheet row to the flat webhook payload."""
    return {
        "userMessage":        _str(row.get("userMessage")),
        "LastAImessage":      _str(row.get("LastAImessage")),
        "userMessageSummary": _str(row.get("userMessageSummary")),
        "narrative":          _str(row.get("narrative")),
        "preference":         _str(row.get("preference")),
        "maxPayment":         _str(row.get("maxPayment"), default="Not provided"),
        "ageRange":           _str(row.get("ageRange")),
        "employmentType":     _str(row.get("employmentType")),
        "monthsAsCustomer":   int(_num(row.get("monthsAsCustomer"))),
        "dpd":                int(_num(row.get("dpd"))),
        "sumOsNCB":           _num(row.get("sumOsNCB")),
        "installmentNCB_Y1":  _num(row.get("installmentNCB_Y1")),
        "installmentNCB_Y2":  _num(row.get("installmentNCB_Y2")),
        "installmentNCB_Y3":  _num(row.get("installmentNCB_Y3")),
        "offerReadableText":  _str(
            row.get("offerReadableText"),
            default="The user does not select the offer suggested by AI",
        ),
    }


def call_summary(row: dict, timeout: int = TIMEOUT, retries: int = RETRIES) -> dict:
    payload  = build_payload(row)
    url      = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/515736a7-e9eb-4ea0-81cb-bfd4a4248a8b


In [4]:
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)

Loaded 250 rows  |  columns: ['testId', 'inputPath', 'userMessage', 'LastAImessage', 'userMessageSummary', 'narrative', 'preference', 'maxPayment', 'ageRange', 'employmentType', 'monthsAsCustomer', 'dpd', 'sumOsNCB', 'installmentNCB_Y1', 'installmentNCB_Y2', 'installmentNCB_Y3', 'offerReadableText', 'situation_tested']


,testId,inputPath,userMessage,LastAImessage,userMessageSummary,narrative,preference,maxPayment,ageRange,employmentType,monthsAsCustomer,dpd,sumOsNCB,installmentNCB_Y1,installmentNCB_Y2,installmentNCB_Y3,offerReadableText,situation_tested
0,TC-0001,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,"The customer, whose financial situation was no...","{""DebtSituation"":""DebtBurden""}",Not provided,25-29 years old,อื่นๆ (Others),38,38,1021682.50,14750.0,10750.00,6350.00,แผน: TDR ด้วยอัตรากำลังผ่อนชำระของลูกค้า\nที่ม...,รับข้อเสนอ TDR ตามกำลังผ่อน
1,TC-0002,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,"The customer, aiming for a debt restructuring ...",NaN,Not provided,55-59 years old,อาชีพอิสระ/เกษตรกร (Self-Employed/Farmer),50,17,2505376.67,21800.0,14666.67,8966.67,แผน: TDR ด้วยอัตรากำลังผ่อนชำระของลูกค้า\nที่ม...,ขอเจ้าหน้าที่ติดต่อกลับ
2,TC-0003,webhook,The user select one of the offer provided by AI,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,message 1: สนใจแผนที่เสนอครับ,"The customer, whose financial situation led to...","{""DebtSituation"":""DebtBurden""}",Not provided,25-29 years old,อื่นๆ (Others),38,38,1021682.50,14750.0,10750.00,6350.00,แผน: TDR ด้วยอัตรากำลังผ่อนชำระของลูกค้า\nที่ม...,รายได้ลดลงชั่วคราว


In [5]:
RESULT_FIELDS = ["subject", "objective", "debt_cause", "offer_suitability", "request_information", "summary"]

result_buckets = {f: [] for f in RESULT_FIELDS}
errors = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Calling n8n"):
    try:
        resp = call_summary(row.to_dict())
        for field in RESULT_FIELDS:
            result_buckets[field].append(resp.get(field, ""))
        errors.append(None)
    except Exception as exc:
        for field in RESULT_FIELDS:
            result_buckets[field].append(None)
        errors.append(str(exc))
    time.sleep(DELAY_BETWEEN)

for field in RESULT_FIELDS:
    df[f"result_{field}"] = result_buckets[field]
df["error"] = errors

failed = df["error"].notna().sum()
print(f"Done. {len(df) - failed}/{len(df)} succeeded, {failed} failed.")

Calling n8n:   0%|          | 0/250 [00:00<?, ?it/s]

Done. 250/250 succeeded, 0 failed.


In [6]:
# Preview failed rows (if any)
if failed:
    display(df[df["error"].notna()][["testId", "situation_tested", "error"]])

In [7]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
result_cols = [f"result_{f}" for f in RESULT_FIELDS]
df[["testId", "situation_tested"] + result_cols].head(5)

Saved → summary_result.xlsx


,testId,situation_tested,result_subject,result_objective,result_debt_cause,result_offer_suitability,result_request_information,result_summary
0,TC-0001,รับข้อเสนอ TDR ตามกำลังผ่อน,ขอปรับโครงสร้างหนี้ตามความสามารถในการชำระ,วัตถุประสงค์คือการปรับโครงสร้างหนี้สินเชื่อ 2 ...,ลูกค้ามีภาระหนี้สูงจากยอดหนี้ภายนอกธนาคารจำนวน...,ข้อเสนอนี้มีความเหมาะสมกับสถานการณ์ของลูกค้า เ...,ไม่มีข้อมูลเพิ่มเติม,ลูกค้ามีอายุระหว่าง 25-29 ปี เป็นลูกค้ามา 38 เ...
1,TC-0002,ขอเจ้าหน้าที่ติดต่อกลับ,ขอปรับโครงสร้างหนี้บัญชีสินเชื่อ xxxxxx503379,ดำเนินการอนุมัติแผนปรับโครงสร้างหนี้ (TDR) สำห...,ลูกหนี้มีภาระหนี้สูงจากสินเชื่อภายนอกธนาคาร รว...,ข้อเสนอนี้เหมาะสมกับสถานการณ์ของลูกค้าซึ่งมีภา...,ไม่มีข้อมูลเพิ่มเติม,ลูกค้าเป็นอาชีพอิสระ มีอายุ 55-59 ปี และเป็นลู...
2,TC-0003,รายได้ลดลงชั่วคราว,ขอปรับโครงสร้างหนี้ตามแผน TDR ที่ลูกค้าเลือก,วัตถุประสงค์คือการดำเนินการปรับโครงสร้างหนี้ (...,ลูกค้ารายนี้ประสบปัญหาภาระหนี้สูงเกินกำลัง โดย...,ข้อเสนอ TDR นี้เหมาะสมกับสถานการณ์ของลูกค้า เน...,ไม่มีข้อมูลเพิ่มเติม,ลูกค้ารายนี้เป็นพนักงานประเภทอื่นๆ อายุระหว่าง...
3,TC-0004,เปลี่ยนอาชีพ,ขอปรับโครงสร้างหนี้ตามแผน TDR สำหรับลูกค้า,วัตถุประสงค์เพื่อปรับโครงสร้างหนี้ของลูกค้าจำน...,ลูกค้ามีภาระหนี้สูงจากยอดหนี้คงค้างภายนอกธนาคา...,ข้อเสนอนี้มีความเหมาะสมอย่างยิ่งกับสถานการณ์ทา...,ไม่มีข้อมูลเพิ่มเติม,ลูกค้าเป็นพนักงานประจำ อายุ 55-59 ปี มีภาระหนี...
4,TC-0005,ภาระหนี้หลายสถาบัน,ขอปรับโครงสร้างหนี้สินเชื่อบุคคล,คำร้องนี้มีวัตถุประสงค์เพื่อขอปรับโครงสร้างหนี...,ลูกหนี้เป็นพนักงานประจำที่มีภาระหนี้ภายนอกสูงก...,ข้อเสนอ TDR นี้เหมาะสมกับสถานการณ์ของลูกหนี้ เ...,ไม่มีข้อมูลเพิ่มเติม,ลูกหนี้เป็นพนักงานประจำที่ประสบปัญหาภาระหนี้สู...
